In [297]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import os
import requests
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [298]:
uri = os.getenv('SBS_V1_MONGO_URI')

client = MongoClient(uri, server_api=ServerApi('1'))
db = client['SBSV1']

try:
    client.admin.command('ping')
    print('Pinged your deployment. You successfully connected to MongoDB!')
except Exception as e:
    print(e)

nba_games_historical_collection = db['nba_games_historical']
nba_team_aggregated_game_stats_historical_collection = db['nba_team_aggregated_game_stats_historical']
nba_game_player_stats_historical_collection = db['nba_game_player_stats_historical']
nba_player_aggregated_game_stats_historical_collection = db['nba_player_aggregated_game_stats_historical']
cached_web_api_response_collection = db['cached_web_api_response']

Pinged your deployment. You successfully connected to MongoDB!


In [299]:
############# SPORTS BETTING SANDBOX API ################

#########################################################
# get_event_odds ########################################
def get_event_odds(sports):
    url = 'https://sportsbettingsandboxapi.com/odds-api/events/get'
    response = requests.post(url, json={ 'sports': sports }).json()['data']
    return response
#########################################################

#########################################################
# get_event_odds ########################################
#[derive(Debug, Deserialize, Clone)]
#[serde(rename_all = "camelCase")]
# pub struct GetOddsRequest {
#     pub sports: OddsApiSports,
#     pub regions: OddsApiRegions,
#     pub markets: Vec<String>,
#     pub odds_format: OddsFormat,
#     pub bookmakers: Vec<Bookmakers>
# }
def get_odds(req):
    url = 'https://sportsbettingsandboxapi.com/odds-api/odds/get'
    response = requests.post(url, json=req).json()['data']['events']
    return response
#########################################################

In [ ]:
################### MONGO FUNCS #########################

#########################################################
# get_historical_nba_game_objs_from_season ##############
def get_historical_nba_game_objs_from_season(season):
    return list(nba_games_historical_collection.find({ 'season': season }))
#########################################################



#########################################################
# get_nba_player_aggregated_games_stats_from_season #####

# TODO FINISH THIS:: add playerId, teamId, opponentId, season to feature map
def get_nba_player_aggregated_games_stats_from_season(season):
    player_objs = list(nba_player_aggregated_game_stats_historical_collection.find({ 'season': season, 'seasonType': 'ALL' }))
    
    for player in player_objs:
        if player['playerId'] in game_stats_per_player:
            game_stats_per_player[player['playerId']] = game_stats_per_player[player['playerId']] + list(player['playerStats'].values())
        else:
            player_stats_df = pd.DataFrame(list(player['playerStats'].values()))
            player_stats_df['playerId'] = player['playerId']
            player_stats_df['teamId'] = player.['teamId']

            
            game_stats_per_player[player['playerId']] = list(player['playerStats'].values())    
#########################################################

#########################################################
# get_enriched_player_stats_from_season #################
def get_enriched_player_stats_from_season(historical_game_objs, historical_player_stats, ):
#########################################################

In [300]:
##################### ML FUNCS ##########################

#########################################################
# get_nba_player_type_mapping ###########################
def get_nba_player_type_mapping(season):
    player_stats_for_clustering = ["points", "assists", "totReb", "fgm", "fga", 
    "tpm", "tpa", "ftm", "fta", "turnovers", "blocks", "steals"]
    
    game_stats_per_player = dict()
    player_objs = list(nba_player_aggregated_game_stats_historical_collection.find({ 'season': season, 'seasonType': 'ALL' }))
    
    for player in player_objs:
        if player['playerId'] in game_stats_per_player:
            game_stats_per_player[player['playerId']] = game_stats_per_player[player['playerId']] + list(player['playerStats'].values())
        else:
            game_stats_per_player[player['playerId']] = list(player['playerStats'].values())

    all_players_avg_stats = []
    for player_id, game_stats in game_stats_per_player.items():
        player_games_stats = pd.DataFrame(game_stats)
        player_games_stats = player_games_stats[player_games_stats['min'] > 0]

        for stat in player_stats_for_clustering:
            player_games_stats[stat] = player_games_stats[stat] * (36 / player_games_stats['min']) 
        
        player_games_stats = player_games_stats.sort_values(by="dateStart", ascending=False)[player_stats_for_clustering]
            
        player_avg_stats = player_games_stats.mean()
        player_avg_stats['playerId'] = player_id
        all_players_avg_stats.append(player_avg_stats)

    all_players_avg_stats = pd.DataFrame(all_players_avg_stats)

    all_players_avg_stats = all_players_avg_stats.dropna()

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(all_players_avg_stats.drop("playerId", axis=1))

    kmeans = KMeans(n_clusters=5, random_state=42)
    all_players_avg_stats["role_cluster"] = kmeans.fit_predict(X_scaled)
    
    player_id_and_cluster_val_df = all_players_avg_stats[['playerId', 'role_cluster']]
    return player_id_and_cluster_val_df
#########################################################

In [301]:
player_cluster_mapping_all_games = get_nba_player_type_mapping(2024, None)
player_cluster_mapping_last_10_games = get_nba_player_type_mapping(2024, 10)

print(player_cluster_mapping_all_games)
print(player_cluster_mapping_last_10_games)

     playerId  role_cluster
0      3415.0             0
1      3938.0             3
2        92.0             0
3      2327.0             1
4      1868.0             3
..        ...           ...
556    2853.0             2
557    4150.0             2
558    3999.0             1
559    3437.0             1
560    4137.0             2

[558 rows x 2 columns]
     playerId  role_cluster
0      3415.0             2
1      3938.0             4
2        92.0             2
3      2327.0             1
4      1868.0             1
..        ...           ...
556    2853.0             3
557    4150.0             3
558    3999.0             4
559    3437.0             1
560    4137.0             3

[558 rows x 2 columns]


In [302]:
p = [1,2,4]


ValueError: 3 is not in list